# 第 2 章互动实验：卷积、梯度与边缘

本实验只用 OpenCV 作为观察基线，不给出 `2-1_手写相关运算.py` 的实现。数据由代码合成，不依赖网络。先预测噪声、平滑尺度和阈值如何改变边缘，再拖动滑块验证。

In [ ]:
%matplotlib inline
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

print('OpenCV:', cv2.__version__)

In [ ]:
def make_scene(noise_std=10):
    image = np.full((128, 160), 35, dtype=np.float32)
    image[28:100, 35:125] = 170
    cv2.line(image, (12, 112), (148, 18), 235, 2)
    rng = np.random.default_rng(2026)
    image += rng.normal(0, noise_std, image.shape)
    return np.clip(image, 0, 255).astype(np.uint8)

def show_edges(noise_std=10, sigma=1.0, low=50, high=120):
    image = make_scene(noise_std)
    k = max(3, int(np.ceil(6 * sigma)) | 1)
    smooth = cv2.GaussianBlur(image, (k, k), sigma)
    gx = cv2.Sobel(smooth, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(smooth, cv2.CV_32F, 0, 1, ksize=3)
    magnitude = cv2.magnitude(gx, gy)
    edges = cv2.Canny(smooth, low, max(high, low + 1), L2gradient=True)
    fig, axes = plt.subplots(1, 4, figsize=(15, 3.6))
    for ax, data, title, cmap in zip(
        axes, [image, gx, magnitude, edges],
        ['合成图', '带符号 Gx', '梯度幅值', f'Canny：{np.count_nonzero(edges)} 点'],
        ['gray', 'coolwarm', 'magma', 'gray']):
        ax.imshow(data, cmap=cmap)
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout(); plt.show()

interact(
    show_edges,
    noise_std=IntSlider(min=0, max=40, step=5, value=10),
    sigma=FloatSlider(min=0.5, max=3.0, step=0.5, value=1.0),
    low=IntSlider(min=10, max=150, step=10, value=50),
    high=IntSlider(min=40, max=240, step=10, value=120),
);

## 破坏实验与记录

1. 把噪声调到 40 且平滑降到 0.5，记录误检。
2. 把平滑提高到 3.0，观察细斜线定位和连续性。
3. 让低阈值接近高阈值，观察弱边是否断裂。

把参数、边缘像素数、失败原因写入 `实验记录.md`，再独立完成作业。